# 11-class Strong Differential Attention Comparison

This notebook trains and compares three 11-class AMR models on the same RadioML2016.10a split:

1. Baseline MCLDNN with 2?LSTM-128
2. Strong MCLDNN attention
3. Strong MCLDNN differential attention

The strong differential-attention model keeps the stronger attention scaffold:

- 3-branch MCLDNN CNN front-end
- 100 ? 128 sequence projection
- sinusoidal positional encoding
- two residual differential-attention blocks
- gated temporal pooling + mean/max pooling
- larger regularized classifier head

Goal: test whether differential-attention cancellation improves over the strong normal-attention model while keeping the same 11-class split and evaluation protocol.


In [ ]:
# CELL 1: Setup repo and paths
import os
import sys
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, FileLink, HTML

os.environ['KERAS_BACKEND'] = 'tensorflow'

REPO_URL = 'https://github.com/akshlabh/amr-5-class.git'
WORK_DIR = Path('/kaggle/working/amr-5-class')

DATASET_CANDIDATES = [
    Path('/kaggle/input/datasets/gustavopolicarpo/rml201610a-dict/RML2016.10a_dict.dat'),
    Path('/kaggle/input/rml201610a-dict/RML2016.10a_dict.dat'),
    Path('/kaggle/input/radioml2016-10a/RML2016.10a_dict.pkl'),
    Path('/kaggle/input/radioml2016-10a/RML2016.10a_dict.dat'),
    Path('data/RML2016.10a_dict.pkl'),
    Path('data/RML2016.10a_dict.dat'),
]

def find_attached_repo():
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return None
    for root in input_root.glob('**'):
        if (root / 'src' / 'train.py').exists() and (root / 'configs').exists():
            return root
    return None

if (Path.cwd() / 'src' / 'train.py').exists():
    WORK_DIR = Path.cwd()
    print('Using current repo:', WORK_DIR)
elif (WORK_DIR / 'src' / 'train.py').exists():
    print('Using existing repo:', WORK_DIR)
    try:
        subprocess.run(['git', '-C', str(WORK_DIR), 'pull'], check=False)
    except Exception as e:
        print('Git pull skipped:', e)
else:
    attached = find_attached_repo()
    if attached is not None:
        print('Copying attached repo from:', attached)
        if WORK_DIR.exists():
            shutil.rmtree(WORK_DIR)
        shutil.copytree(attached, WORK_DIR)
    else:
        print('Cloning repo from GitHub...')
        subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

DATASET = next((p for p in DATASET_CANDIDATES if p.exists()), None)
assert DATASET is not None, 'Dataset not found. Set DATASET manually in this cell.'

BASELINE_DIR = Path('experiments/11class_baseline')
STRONG_ATT_DIR = Path('experiments/11class_attention_strong')
STRONG_DIFF_DIR = Path('experiments/11class_diffattention_strong')
COMPARE_DIR = Path('experiments/11class_diffattention_strong_comparison')
FIG_DIR = COMPARE_DIR / 'figures'
RES_DIR = COMPARE_DIR / 'results'
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

print('Working dir:', Path.cwd())
print('Dataset    :', DATASET)
print('Dataset OK :', DATASET.exists())


In [ ]:
# CELL 2: Verify required files and parameter counts
import keras
keras.mixed_precision.set_global_policy('float32')

required = [
    'src/train.py',
    'src/dataset.py',
    'src/models/mcldnn.py',
    'src/models/mcldnn_attention_strong.py',
    'src/models/mcldnn_diffattention_strong.py',
    'configs/exp_11class_baseline.yaml',
    'configs/exp_11class_attention_strong.yaml',
    'configs/exp_11class_diffattention_strong.yaml',
]
for f in required:
    print(('OK      ' if Path(f).exists() else 'MISSING ') + f)
    assert Path(f).exists(), f'Missing required file: {f}'

from src.models.mcldnn import MCLDNN
from src.models.mcldnn_attention_strong import build_mcldnn_attention_strong
from src.models.mcldnn_diffattention_strong import build_mcldnn_diffattention_strong

models_for_count = {
    'Baseline MCLDNN LSTM': MCLDNN(classes=11),
    'Strong attention': build_mcldnn_attention_strong(classes=11),
    'Strong differential attention': build_mcldnn_diffattention_strong(classes=11),
}

print('
Parameter counts')
print('-' * 60)
for name, model in models_for_count.items():
    print(f'{name:<36} {model.count_params():>12,}')
    del model
keras.backend.clear_session()


In [ ]:
# CELL 3: Train missing models
# Existing checkpoints are reused. Missing checkpoints are trained.

def run_stream(cmd):
    print('Running:', ' '.join(map(str, cmd)))
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    rc = process.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, process.args)

def checkpoint_ready(exp_dir):
    return (Path(exp_dir) / 'checkpoints/best_model.weights.h5').exists()

jobs = [
    ('11class_baseline_lstm', BASELINE_DIR, 'configs/exp_11class_baseline.yaml'),
    ('11class_strong_attention', STRONG_ATT_DIR, 'configs/exp_11class_attention_strong.yaml'),
    ('11class_strong_diffattention', STRONG_DIFF_DIR, 'configs/exp_11class_diffattention_strong.yaml'),
]

for name, exp_dir, cfg in jobs:
    if checkpoint_ready(exp_dir):
        print(f'{name}: checkpoint found, skipping training -> {exp_dir}/checkpoints/best_model.weights.h5')
    else:
        print(f'{name}: checkpoint missing, training now...')
        run_stream([sys.executable, '-u', 'src/train.py', '--config', cfg, '--datasetpath', str(DATASET)])
        assert checkpoint_ready(exp_dir), f'Training finished but checkpoint missing: {exp_dir}'


In [ ]:
# CELL 4: Load result CSVs and build comparison tables
models = {
    'Baseline LSTM': {
        'dir': BASELINE_DIR,
        'color': '#9467bd',
        'marker': 'x',
    },
    'Strong attention': {
        'dir': STRONG_ATT_DIR,
        'color': '#1f77b4',
        'marker': 'o',
    },
    'Strong diff-attention': {
        'dir': STRONG_DIFF_DIR,
        'color': '#d62728',
        'marker': 's',
    },
}

rows = []
for name, cfg in models.items():
    score_csv = cfg['dir'] / 'results/test_score.csv'
    snr_csv = cfg['dir'] / 'results/acc_per_snr.csv'
    assert score_csv.exists(), f'Missing {score_csv}'
    assert snr_csv.exists(), f'Missing {snr_csv}'

    score = pd.read_csv(score_csv)
    snr_df = pd.read_csv(snr_csv)
    acc_col = 'accuracy' if 'accuracy' in score.columns else score.columns[-1]

    test_acc = float(score[acc_col].iloc[0])
    mean_snr_acc = float(snr_df['accuracy'].mean())
    low_snr_acc = float(snr_df[snr_df['snr'] <= 0]['accuracy'].mean())
    high_snr_acc = float(snr_df[snr_df['snr'] >= 2]['accuracy'].mean())

    rows.append({
        'model': name,
        'test_accuracy': test_acc,
        'mean_snr_accuracy': mean_snr_acc,
        'low_snr_mean_acc_snr_le_0': low_snr_acc,
        'high_snr_mean_acc_snr_ge_2': high_snr_acc,
        'results_dir': str(cfg['dir']),
    })

summary = pd.DataFrame(rows)
summary.to_csv(RES_DIR / '11class_strong_diffattention_model_summary.csv', index=False)
print('Summary')
display(summary)
print('Saved:', RES_DIR / '11class_strong_diffattention_model_summary.csv')

snr_table = None
for name, cfg in models.items():
    snr_df = pd.read_csv(cfg['dir'] / 'results/acc_per_snr.csv')[['snr', 'accuracy']]
    snr_df = snr_df.rename(columns={'accuracy': name})
    snr_table = snr_df if snr_table is None else snr_table.merge(snr_df, on='snr', how='outer')

snr_table = snr_table.sort_values('snr')
snr_table['Strong diff minus strong attention'] = snr_table['Strong diff-attention'] - snr_table['Strong attention']
snr_table['Strong diff minus LSTM'] = snr_table['Strong diff-attention'] - snr_table['Baseline LSTM']
snr_table.to_csv(RES_DIR / '11class_strong_diffattention_acc_per_snr_comparison.csv', index=False)
display(snr_table)
print('Saved:', RES_DIR / '11class_strong_diffattention_acc_per_snr_comparison.csv')


In [ ]:
# CELL 5: Plot accuracy vs SNR and delta curves
snrs = snr_table['snr'].to_numpy()

plt.figure(figsize=(12.5, 6.4))
for name, cfg in models.items():
    ys = snr_table[name].to_numpy() * 100
    plt.plot(snrs, ys, marker=cfg['marker'], linewidth=2.5, color=cfg['color'], label=name)

plt.xlabel('SNR (dB)')
plt.ylabel('Test Accuracy (%)')
plt.title('11-class Comparison: LSTM vs Strong Attention vs Strong Differential Attention')
plt.grid(True, alpha=0.3)
plt.xticks(snrs)
plt.legend()
plt.tight_layout()
acc_plot = FIG_DIR / '11class_lstm_strong_attention_strong_diffattention_acc_vs_snr.png'
plt.savefig(acc_plot, dpi=220, bbox_inches='tight')
plt.close()
display(Image(filename=str(acc_plot)))
print('Saved:', acc_plot)

plt.figure(figsize=(12, 5))
plt.plot(snrs, snr_table['Strong diff minus strong attention'].to_numpy() * 100,
         marker='s', linewidth=2.4, label='Strong diff-attention - strong attention')
plt.plot(snrs, snr_table['Strong diff minus LSTM'].to_numpy() * 100,
         marker='x', linewidth=2.4, label='Strong diff-attention - baseline LSTM')
plt.axhline(0, color='black', linewidth=1)
plt.xlabel('SNR (dB)')
plt.ylabel('Delta accuracy (percentage points)')
plt.title('11-class Strong Differential Attention Delta')
plt.grid(True, alpha=0.3)
plt.xticks(snrs)
plt.legend()
plt.tight_layout()
delta_plot = FIG_DIR / '11class_strong_diffattention_delta_vs_baselines.png'
plt.savefig(delta_plot, dpi=220, bbox_inches='tight')
plt.close()
display(Image(filename=str(delta_plot)))
print('Saved:', delta_plot)


In [ ]:
# CELL 6: Plot training histories and confusion matrices
for name, cfg in models.items():
    log_path = cfg['dir'] / 'logs/training_log.csv'
    if not log_path.exists():
        print('Missing training log:', log_path)
        continue
    log = pd.read_csv(log_path)
    epoch = log['epoch'] if 'epoch' in log.columns else np.arange(1, len(log) + 1)
    acc_col = next((c for c in log.columns if 'accuracy' in c and 'val' not in c), 'accuracy')
    val_acc_col = next((c for c in log.columns if 'val' in c and 'accuracy' in c), 'val_accuracy')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f'{name} training history', fontweight='bold')
    axes[0].plot(epoch, log['loss'], label='train loss')
    axes[0].plot(epoch, log['val_loss'], label='val loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(epoch, log[acc_col] * 100, label='train acc')
    axes[1].plot(epoch, log[val_acc_col] * 100, label='val acc')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    plt.tight_layout()

    safe = name.lower().replace(' ', '_').replace('-', '_')
    out = FIG_DIR / f'{safe}_training_history.png'
    plt.savefig(out, dpi=180, bbox_inches='tight')
    plt.close()
    display(Image(filename=str(out)))
    print('Saved:', out)

for name, cfg in models.items():
    cm_path = cfg['dir'] / 'figures/confusion_all_snrs.png'
    if cm_path.exists():
        print('
', name, 'confusion matrix')
        display(Image(filename=str(cm_path)))


In [ ]:
# CELL 7: Create repo-ready zip for strong differential-attention comparison
stamp = datetime.now().strftime('%Y%m%d_%H%M')
zip_name = f'11class_strong_diffattention_comparison_repo_ready_{stamp}'
staging_dir = Path('/kaggle/working') / f'{zip_name}_staging'
zip_base = Path('/kaggle/working') / zip_name
zip_path = Path('/kaggle/working') / f'{zip_name}.zip'

if staging_dir.exists():
    shutil.rmtree(staging_dir)
if zip_path.exists():
    zip_path.unlink()

folders_to_zip = [
    'experiments/11class_baseline',
    'experiments/11class_attention_strong',
    'experiments/11class_diffattention_strong',
    'experiments/11class_diffattention_strong_comparison',
]

for rel_folder in folders_to_zip:
    src = WORK_DIR / rel_folder
    dst = staging_dir / rel_folder
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Included: {rel_folder}')
    else:
        print(f'WARNING: Missing, not included: {rel_folder}')

created_zip = shutil.make_archive(str(zip_base), 'zip', root_dir=str(staging_dir))
print('
Created repo-ready zip:')
print(created_zip)
print('
Extract this at repo root. It will create/update:')
for rel_folder in folders_to_zip:
    print(f'  {rel_folder}/')

simple_zip = Path('/kaggle/working/results.zip')
if simple_zip.exists():
    simple_zip.unlink()
shutil.copy2(created_zip, simple_zip)
print('
Simple download copy:', simple_zip)
print('If direct link fails, use Kaggle right sidebar: Output -> /kaggle/working -> results.zip')

display(FileLink(str(simple_zip)))
display(HTML(f'<a href="files/{simple_zip.name}" target="_blank" download>Download results.zip</a>'))
